# Short-List Promising Models & Fine-Tune the System

In [39]:
import pickle
import numpy as np
import pandas as pd

from sklearn import set_config
set_config(transform_output="pandas")

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

In [57]:
# Copied from the Preparation notebook (needed to load preprocessor.pkl)
class EarlySpringFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, add_deltas=True, add_pre_march_trend=True, combine_oni=True, drop_redundant=True, use_year=False,
                 delta_features=("soil_temp_mean", "farm_avg_temp_mean", "om_temp_mean"),
                 redundant_features=("soil_thawed_days_jan", "soil_thawed_days_feb", "farm_freezing_days_feb")):
        self.add_deltas = add_deltas
        self.add_pre_march_trend = add_pre_march_trend
        self.combine_oni = combine_oni
        self.drop_redundant = drop_redundant
        self.use_year = use_year
        self.delta_features = delta_features
        self.redundant_features = redundant_features

    def fit(self, X, y=None):
        self.feature_names_out_ = self._engineer(X).columns.to_numpy()
        return self

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_out_

    def transform(self, X):
        return self._engineer(X)

    def _engineer(self, X):
        data = X.drop(columns="early_spring", errors="ignore")
        data["phil_pred"] = data["phil_pred"].astype(int)

        if self.add_deltas:
            for name in self.delta_features:
                data[f"{name}_delta"] = data[f"{name}_feb"] - data[f"{name}_jan"]

        if self.add_pre_march_trend:
            for prefix in ["farm_avg_temp", "soil_mean_soil_temp", "om_temperature_2m_mean"]:
                d1 = f"{prefix}_d_minus_1"
                d7 = f"{prefix}_d_minus_7"
                if d1 in data.columns and d7 in data.columns:
                    data[f"{prefix}_7d_trend"] = data[d1] - data[d7]

        if self.combine_oni:
            data["oni_winter_avg"] = data[["oni_prev_nov", "oni_prev_dec"]].mean(axis=1)
            data = data.drop(columns=["oni_prev_nov", "oni_prev_dec"])

        if self.drop_redundant:
            data = data.drop(columns=list(self.redundant_features), errors="ignore")

        if self.use_year:
            data["year"] = data.index.to_numpy()

        return data

In [41]:
train_set = pd.read_parquet("train_set.parquet")
X = train_set.drop(columns="early_spring")
y = train_set["early_spring"].astype(int)

with open("preprocessor.pkl", "rb") as f:
    preprocessor = pickle.load(f)

X_prepared = preprocessor.fit_transform(X, y)
print(X.shape, "->", X_prepared.shape)
y.value_counts()

(57, 91) -> (57, 113)


early_spring
0    43
1    14
Name: count, dtype: int64

Early Spring is really rare, so it is apparent that accuracy will be a very misleading metric. We will want to focus on recall and precision.

In [42]:
# Stratified so every fold has some early springs
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

For our baseline, we will be looking at Phil and his predictions. He has an accuracy of 72% precision of 40% and recall of 29% with an F1 score of 33%. Always early would have a recall of 1 but an accuracy of about 25%. Always late would have an accuracy of 75% and a recall of 0. We can get both sets of these metrics by just simply looking at the distribution of the Early Spring target feature.

In [43]:
# Baseline: Phil's predictions
phil = X["phil_pred"].astype(int)
accuracy  = accuracy_score(y, phil)
precision = precision_score(y, phil)
recall    = recall_score(y, phil)
f1        = f1_score(y, phil)
print(f"Phil - accuracy: {accuracy:.3f}  precision: {precision:.3f}  recall: {recall:.3f}  f1: {f1:.3f}")

Phil - accuracy: 0.719  precision: 0.400  recall: 0.286  f1: 0.333


We ended up training 10 models from different families: logistic regression and SGD (linear), KNN, Gaussian naive Bayes, decision tree, random forest and gradient boosting (trees), linear and RBF SVMs, and an MLP, all with default-style parameters. Where a model supports it we used class_weight="balanced" so the rare early springs count for more.

In [44]:
# class_weight="balanced" makes the models predict early spring more often (higher recall)
lr = LogisticRegression(class_weight="balanced", max_iter=5000)
lr.fit(X_prepared, y)
preds = lr.predict(X_prepared)
cv_preds = cross_val_predict(lr, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 1.000  precision: 1.000  recall: 1.000  f1: 1.000
CV    - accuracy: 0.544  precision: 0.167  recall: 0.214  f1: 0.188


array([[28, 15],
       [11,  3]])

It fits the training data perfectly but drops to CV accuracy 0.54 and F1 0.19. It has 15 false alarms and catches only 3 early springs, so it’s worse than Phil. This is overfitting: default regularization is too weak for 113 features and 57 rows.

In [45]:
sgd = SGDClassifier(class_weight="balanced", random_state=42)
sgd.fit(X_prepared, y)
preds = sgd.predict(X_prepared)
cv_preds = cross_val_predict(sgd, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 1.000  precision: 1.000  recall: 1.000  f1: 1.000
CV    - accuracy: 0.561  precision: 0.176  recall: 0.214  f1: 0.194


array([[29, 14],
       [11,  3]])

Almost identical to logistic regression: perfect training scores, CV F1 0.19, 14 false alarms and 3 early springs caught. The same overfitting problem applies, since it’s also a linear model.

In [46]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_prepared, y)
preds = knn.predict(X_prepared)
cv_preds = cross_val_predict(knn, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 0.772  precision: 0.600  recall: 0.214  f1: 0.316
CV    - accuracy: 0.649  precision: 0.000  recall: 0.000  f1: 0.000


array([[37,  6],
       [14,  0]])

It never predicted early spring in CV (recall 0, F1 0) and just predicts “late” for almost every year, which gives accuracy 0.65 only because late years are common. With many noisy features, “nearest” years are not meaningfully similar. This one is not useful.

In [47]:
nb = GaussianNB()
nb.fit(X_prepared, y)
preds = nb.predict(X_prepared)
cv_preds = cross_val_predict(nb, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 0.386  precision: 0.286  recall: 1.000  f1: 0.444
CV    - accuracy: 0.404  precision: 0.206  recall: 0.500  f1: 0.292


array([[16, 27],
       [ 7,  7]])

It predicts early spring very often: recall 0.50 but accuracy only 0.40, with 27 false alarms. That’s close to the always-early behavior and would cost farmers a lot of wasted plantings. Its independence assumption is badly violated by correlated weather features.

In [48]:
dt = DecisionTreeClassifier(class_weight="balanced", min_samples_leaf=3, random_state=42)
dt.fit(X_prepared, y)
preds = dt.predict(X_prepared)
cv_preds = cross_val_predict(dt, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 0.930  precision: 0.778  recall: 1.000  f1: 0.875
CV    - accuracy: 0.667  precision: 0.368  recall: 0.500  f1: 0.424


array([[31, 12],
       [ 7,  7]])

Best base F1 at 0.42, with recall 0.50 and precision 0.37. It catches 7 early springs but has 12 false alarms. It beats Phil, though the training score of 0.93 versus CV of 0.67 shows a single tree is unstable.

In [49]:
rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf.fit(X_prepared, y)
preds = rf.predict(X_prepared)
cv_preds = cross_val_predict(rf, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 1.000  precision: 1.000  recall: 1.000  f1: 1.000
CV    - accuracy: 0.737  precision: 0.455  recall: 0.357  f1: 0.400


array([[37,  6],
       [ 9,  5]])

Best accuracy at 0.74, with precision 0.45, recall 0.36 and F1 0.40. It has only 6 false alarms (the same as Phil) while catching 5 early springs against Phil’s 4. Training scores are perfect, so it still overfits, but averaging many trees helps. This is the most balanced base model.

In [50]:
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_prepared, y)
preds = gb.predict(X_prepared)
cv_preds = cross_val_predict(gb, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 1.000  precision: 1.000  recall: 1.000  f1: 1.000
CV    - accuracy: 0.632  precision: 0.182  recall: 0.143  f1: 0.160


array([[34,  9],
       [12,  2]])

CV F1 is only 0.16 despite perfect training scores. It catches 2 early springs with 9 false alarms. Boosting fits the training errors closely, which on this little data means memorizing noise.

In [51]:
svm_linear = SVC(kernel="linear", class_weight="balanced", random_state=42)
svm_linear.fit(X_prepared, y)
preds = svm_linear.predict(X_prepared)
cv_preds = cross_val_predict(svm_linear, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 1.000  precision: 1.000  recall: 1.000  f1: 1.000
CV    - accuracy: 0.561  precision: 0.133  recall: 0.143  f1: 0.138


array([[30, 13],
       [12,  2]])

CV F1 0.14, the worst of the 10. It catches 2 early springs and has 13 false alarms. Like the other linear models, it overfits with default settings.

In [52]:
svm_rbf = SVC(kernel="rbf", class_weight="balanced", random_state=42)
svm_rbf.fit(X_prepared, y)
preds = svm_rbf.predict(X_prepared)
cv_preds = cross_val_predict(svm_rbf, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 0.982  precision: 0.933  recall: 1.000  f1: 0.966
CV    - accuracy: 0.719  precision: 0.333  recall: 0.143  f1: 0.200


array([[39,  4],
       [12,  2]])

Accuracy 0.72 but recall only 0.14 (2 early springs caught), with just 4 false alarms. It’s cautious: it rarely predicts early, which keeps false alarms low but misses most opportunities.

In [53]:
mlp = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=2000, random_state=42)
mlp.fit(X_prepared, y)
preds = mlp.predict(X_prepared)
cv_preds = cross_val_predict(mlp, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 1.000  precision: 1.000  recall: 1.000  f1: 1.000
CV    - accuracy: 0.667  precision: 0.273  recall: 0.214  f1: 0.240


array([[35,  8],
       [11,  3]])

CV F1 0.24, catching 3 early springs with 8 false alarms. A neural net with 57 rows and 113 features has far too many parameters, so it overfits and doesn’t generalize.

In [54]:
for feature, importance in sorted(zip(rf.feature_names_in_, rf.feature_importances_), key=lambda x: x[1], reverse=True):
    print(f"{feature}: {importance:.3f}")

om_precipitation_sum_d_minus_6: 0.050
farm_snowfall_sum_feb: 0.044
om_snowfall_sum_feb: 0.042
farm_avg_temp_d_minus_1: 0.034
om_precip_sum_feb: 0.025
om_temperature_2m_mean_d_minus_1: 0.024
soil_moisture_mean_feb: 0.024
farm_precip_sum_jan: 0.024
soil_mean_soil_temp_d_minus_6: 0.023
soil_mean_soil_temp_d_minus_5: 0.022
farm_snowdepth_mean_feb: 0.019
om_snowfall_sum_jan: 0.019
soil_temp_mean_delta: 0.018
soil_mean_soil_moisture_d_minus_3: 0.017
soil_temp_min_jan: 0.016
soil_mean_soil_temp_d_minus_1: 0.016
om_temperature_2m_mean_d_minus_6: 0.015
om_sunshine_hours_sum_jan: 0.014
farm_snow_cover_days_feb: 0.014
soil_temp_max_jan: 0.014
om_precip_sum_jan: 0.014
soil_temp_mean_jan: 0.014
om_temp_mean_delta: 0.014
om_temperature_2m_mean_d_minus_7: 0.013
om_temp_mean_feb: 0.013
om_temp_min_jan: 0.013
farm_avg_temp_mean_feb: 0.013
soil_temp_min_feb: 0.012
soil_moisture_mean_jan: 0.012
soil_mean_soil_moisture_d_minus_7: 0.012
farm_avg_temp_d_minus_4: 0.012
om_temp_max_jan: 0.012
farm_avg_temp_d_

Importance is spread very thinly: the top feature scores only about 0.05, and most are under 0.02. The top ones are mostly single-day lags like om_precipitation_sum_d_minus_6, and snowfall in February. A single day’s rain a week before March has no clear link to spring, so these likely reflect noise, not real drivers. The honest takeaway is that no variable stands out.

In [55]:
# Positive = early spring, negative = late spring
for feature, coef in sorted(zip(lr.feature_names_in_, lr.coef_[0]), key=lambda x: abs(x[1]), reverse=True):
    print(f"{feature}: {coef:.3f}")

om_precipitation_sum_d_minus_6: -1.105
soil_mean_soil_moisture_d_minus_3: -0.895
soil_moisture_mean_feb: 0.852
farm_max_temp_max_feb: 0.649
soil_mean_soil_moisture_d_minus_7: 0.629
farm_snowfall_sum_feb: -0.603
om_temp_min_feb: -0.535
missingindicator_farm_avg_temp_d_minus_4: 0.492
farm_precip_sum_jan: 0.488
om_temperature_2m_mean_d_minus_6: -0.479
om_snowfall_sum_jan: 0.464
soil_moisture_mean_jan: 0.431
om_precip_sum_jan: 0.427
om_temp_max_feb: 0.397
missingindicator_farm_avg_temp_d_minus_5: -0.396
missingindicator_farm_avg_temp_d_minus_2: -0.378
farm_snowdepth_d_minus_5: 0.362
om_freezing_days_jan: 0.346
om_snowfall_sum_feb: -0.345
om_temp_mean_delta: 0.338
om_temperature_2m_mean_d_minus_1: -0.333
soil_mean_soil_moisture_d_minus_6: 0.326
om_temperature_2m_mean_d_minus_4: 0.325
farm_snowdepth_d_minus_7: 0.306
missingindicator_farm_avg_temp_d_minus_3: 0.305
farm_snowdepth_mean_feb: -0.300
om_temperature_2m_mean_7d_trend: -0.299
om_temp_max_jan: 0.292
missingindicator_farm_snowdepth_d_m

The largest coefficients (around ±1) are again single-day lags, such as precipitation six days before March 1 and soil moisture three days before. Signs flip between neighboring days, which also suggests noise. Unregularized coefficients on correlated features shouldn’t be read as effects. This supports preferring smoothed features and stronger regularization.

We carry forward three single models (logistic regression, an SVM and an SGD classifier), plus the random forest as our tree-based ensemble and a voting ensemble of the three. Their base scores were weak (CV F1 0.14–0.19), but the cause was clear: they fit the training data perfectly with nearly unregularized defaults. Regularization strength, class weights and kernel choice are exactly what tuning changes, so these models had the most to gain, and they represent three different boundary types (a logistic model, a margin-based kernel model, and a linear model trained by stochastic gradient descent). We left out KNN (never predicts early), naive Bayes (nearly always early), and gradient boosting and the MLP (memorize noise with 57 rows). In hindsight the decision tree, with the best base F1, would also have been a reasonable pick. One weakness: logistic regression and SGD made nearly identical errors (15 vs 14 false alarms, 3 early springs caught each), so they are not very diverse, which matters later for the voting ensemble.

In [24]:
import warnings
from scipy.stats import loguniform, randint, uniform
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import make_scorer

# Some random combos of hyperparameters don't converge - hide those warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)
from sklearn.model_selection import RepeatedStratifiedKFold, RandomizedSearchCV, cross_validate

In [25]:
# The log step is a FunctionTransformer inside the skewed pipeline - grab it so "log or not" can be tuned
log_transformer = preprocessor.named_steps["columns"].named_transformers_["skewed"].named_steps["log"]

Each search tunes preprocessing and the model together: derived features (deltas, averaged ONI, redundant columns), imputation strategy and missing-value indicators, and the log transform. Combined with 3–6 model hyperparameters, that gives more than 10 per search (the year feature is fixed off, because trees cannot extrapolate to future years). We use randomized search with 5-fold stratified CV repeated 3 times, because one 5-fold run is too noisy for 57 rows.

We kept F1 as the refit metric, and it has a known flaw: with 25% early springs, “always early” already scores F1 0.39, which beats many sensible models. As a result the SVM search chose exactly that. We therefore judge the finalists on more than F1: precision and false alarms matter, because five wrong early calls in a row ruin profits, and recall matters because missing early springs loses the second harvest. A single wasted planting is acceptable, but a long run of them is not.

In [34]:
# Repeated folds give a more stable score on such a small dataset
tune_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
SCORING = {
    "f1": "f1",
    "recall": "recall",
    "precision": make_scorer(precision_score, zero_division=0),
    "accuracy": "accuracy",
}
REFIT = "f1"  # the metric used to pick the best hyperparameters

# Preprocessing hyperparameters, shared by every search
preprocessing_params = {
    "preprocessor__features__add_deltas": [True, False],
    "preprocessor__features__combine_oni": [True, False],
    "preprocessor__features__drop_redundant": [True, False],
    "preprocessor__features__use_year": [False],
    "preprocessor__columns__numeric__impute__strategy": ["mean", "median"],
    "preprocessor__columns__numeric__impute__add_indicator": [True, False],
    "preprocessor__columns__skewed__impute__strategy": ["mean", "median"],
    "preprocessor__columns__skewed__impute__add_indicator": [True, False],
    "preprocessor__columns__skewed__log": [log_transformer, "passthrough"],
}

def make_search(model, model_params, n_iter=100):
    """Randomized search over preprocessing + model hyperparameters, picking the best REFIT score."""
    pipe = Pipeline([("preprocessor", clone(preprocessor)), ("model", model)])
    return RandomizedSearchCV(pipe, {**preprocessing_params, **model_params}, n_iter=n_iter,
                              cv=tune_cv, scoring=SCORING, refit=REFIT, n_jobs=-1, random_state=42)

def show_best(search):
    """Print the best params and the CV scores of the best model."""
    best = search.best_index_
    for param, value in search.best_params_.items():
        print(f"{param}: {value}")
    print()
    for metric in SCORING:
        print(f"CV {metric}: {search.cv_results_[f'mean_test_{metric}'][best]:.3f}")

In [27]:
svm_search = make_search(SVC(random_state=42), {
    "model__kernel": ["linear", "rbf", "poly"],
    "model__C": loguniform(1e-3, 1e2),
    "model__gamma": ["scale", "auto"],
    "model__degree": [2, 3],
    "model__class_weight": ["balanced", None, {0: 1, 1: 5}],
})
svm_search.fit(X, y)
show_best(svm_search)

model__C: 0.016037155126499166
model__class_weight: {0: 1, 1: 5}
model__degree: 3
model__gamma: scale
model__kernel: poly
preprocessor__columns__numeric__impute__add_indicator: True
preprocessor__columns__numeric__impute__strategy: median
preprocessor__columns__skewed__impute__add_indicator: True
preprocessor__columns__skewed__impute__strategy: median
preprocessor__columns__skewed__log: passthrough
preprocessor__features__add_deltas: False
preprocessor__features__combine_oni: False
preprocessor__features__drop_redundant: False
preprocessor__features__use_year: False

CV f1: 0.393
CV recall: 1.000
CV precision: 0.245
CV accuracy: 0.245


In [28]:
sgd_search = make_search(SGDClassifier(max_iter=5000, random_state=42), {
    "model__loss": ["hinge", "log_loss", "modified_huber"],
    "model__alpha": loguniform(1e-5, 1e0),
    "model__penalty": ["l2", "l1", "elasticnet"],
    "model__l1_ratio": uniform(0, 1),
    "model__class_weight": ["balanced", None, {0: 1, 1: 5}],
})
sgd_search.fit(X, y)
show_best(sgd_search)

model__alpha: 0.10597985905405348
model__class_weight: {0: 1, 1: 5}
model__l1_ratio: 0.13637147558676976
model__loss: log_loss
model__penalty: l1
preprocessor__columns__numeric__impute__add_indicator: False
preprocessor__columns__numeric__impute__strategy: median
preprocessor__columns__skewed__impute__add_indicator: False
preprocessor__columns__skewed__impute__strategy: median
preprocessor__columns__skewed__log: FunctionTransformer(feature_names_out='one-to-one', func=<ufunc 'log1p'>,
                    inverse_func=<ufunc 'expm1'>)
preprocessor__features__add_deltas: True
preprocessor__features__combine_oni: True
preprocessor__features__drop_redundant: True
preprocessor__features__use_year: False

CV f1: 0.406
CV recall: 0.556
CV precision: 0.340
CV accuracy: 0.593


In [29]:
lr_search = make_search(LogisticRegression(solver="saga", max_iter=10000, random_state=42), {
    "model__C": loguniform(1e-3, 1e2),
    "model__l1_ratio": uniform(0, 1),  # 0 = L2, 1 = L1, in between = elastic net
    "model__class_weight": ["balanced", None, {0: 1, 1: 5}],
})
lr_search.fit(X, y)
show_best(lr_search)

model__C: 0.11271562796926876
model__class_weight: {0: 1, 1: 5}
model__l1_ratio: 0.8966102629740208
preprocessor__columns__numeric__impute__add_indicator: False
preprocessor__columns__numeric__impute__strategy: median
preprocessor__columns__skewed__impute__add_indicator: True
preprocessor__columns__skewed__impute__strategy: median
preprocessor__columns__skewed__log: FunctionTransformer(feature_names_out='one-to-one', func=<ufunc 'log1p'>,
                    inverse_func=<ufunc 'expm1'>)
preprocessor__features__add_deltas: False
preprocessor__features__combine_oni: False
preprocessor__features__drop_redundant: True
preprocessor__features__use_year: False

CV f1: 0.449
CV recall: 0.778
CV precision: 0.324
CV accuracy: 0.540


In [37]:
# Ensemble 1: random forest
rf_search = make_search(RandomForestClassifier(random_state=42), {
    "model__n_estimators": randint(50, 500),
    "model__max_depth": [1, 2, 3, 5, 8, None],
    "model__min_samples_leaf": randint(1, 10),
    "model__max_features": ["sqrt", "log2", 0.5, 1.0],
    "model__bootstrap": [True, False],
    "model__class_weight": ["balanced", "balanced_subsample", None],
}, n_iter=50)
rf_search.fit(X, y)
show_best(rf_search)

model__bootstrap: False
model__class_weight: balanced_subsample
model__max_depth: None
model__max_features: 0.5
model__min_samples_leaf: 6
model__n_estimators: 493
preprocessor__columns__numeric__impute__add_indicator: True
preprocessor__columns__numeric__impute__strategy: mean
preprocessor__columns__skewed__impute__add_indicator: False
preprocessor__columns__skewed__impute__strategy: mean
preprocessor__columns__skewed__log: passthrough
preprocessor__features__add_deltas: False
preprocessor__features__combine_oni: False
preprocessor__features__drop_redundant: True
preprocessor__features__use_year: False

CV f1: 0.477
CV recall: 0.489
CV precision: 0.530
CV accuracy: 0.743


Best CV recall 0.49, precision 0.53, F1 0.48, accuracy 0.74, beating Phil on every metric (recall 0.29, precision 0.40, F1 0.33). The search favored heavily regularized trees: min_samples_leaf = 6 (every leaf needs at least six years), max_features = 0.5 (each split sees half the features), 493 trees (the top of the range; more trees just stabilize the average), no bootstrap, and balanced_subsample weights so rare early springs count more. max_depth came back unlimited, even though depth 1 was available, so depth is not the main regularizer here; leaf size is. Preprocessing was simple: mean imputation, no log transform (irrelevant for trees), no deltas or averaged ONI, redundant columns dropped. That is roughly 6 false alarms in 57 years, the same as Phil, while catching about 7 early springs to his 4.

In [31]:
# Ensemble 2: majority vote of the three tuned models (each keeps its own tuned preprocessing)
voting = VotingClassifier([
    ("svm", svm_search.best_estimator_),
    ("sgd", sgd_search.best_estimator_),
    ("lr", lr_search.best_estimator_),
], voting="hard")
voting_scores = cross_validate(voting, X, y, cv=tune_cv, scoring=SCORING, n_jobs=-1)
for metric in SCORING:
    print(f"CV {metric}: {voting_scores[f'test_{metric}'].mean():.3f}")

CV f1: 0.446
CV recall: 0.800
CV precision: 0.317
CV accuracy: 0.511


In [38]:
# Compare everything on the same CV folds
comparison = pd.DataFrame({
    name: {metric: search.cv_results_[f"mean_test_{metric}"][search.best_index_] for metric in SCORING}
    for name, search in [("SVM", svm_search), ("SGD", sgd_search), ("Logistic Regression", lr_search),
                         ("Random Forest", rf_search)]
}).T
comparison.loc["Voting"] = {metric: voting_scores[f"test_{metric}"].mean() for metric in SCORING}
# Baseline: always predict early spring
always_early = cross_validate(DummyClassifier(strategy="constant", constant=1), X, y, cv=tune_cv, scoring=SCORING)
comparison.loc["Always early spring"] = {metric: always_early[f"test_{metric}"].mean() for metric in SCORING}
comparison.sort_values(REFIT, ascending=False).round(3)

,f1,recall,precision,accuracy
Random Forest,0.477,0.489,0.530,0.743
Logistic Regression,0.449,0.778,0.324,0.540
Voting,0.446,0.800,0.317,0.511
SGD,0.406,0.556,0.340,0.593
SVM,0.393,1.000,0.245,0.245
Always early spring,0.393,1.000,0.245,0.245


The random forest has the best F1, precision and accuracy and by far the fewest false alarms. Logistic regression and voting have much higher recall (0.78 and 0.80), but at 0.32 precision they would produce about 23–24 false alarms in 57 years. If false alarms were independent, that gives about a 30–35% chance of a five-in-a-row losing streak somewhere in 57 years, against roughly 1% for the random forest. SVM and always-early are the same model and fail the business rule completely. All the differences are small relative to what 14 early springs can resolve, and the best of 50 to 100 random configurations is optimistic. We can however still say we found a model that modestly beats out Phil's predictions. However, we could probably do good with more advanced tuning that we havent quite covered yet. I also want to note that using the random k folds removes any chance of finding trends in the years with something like climate change possibly attributing to certain results. The years possibly not being independent will most likely interfere with the rest of the model's performance.

In [ ]:


final_model = rf_search.best_estimator_
final_model.fit(X, y)

feat_step = final_model.named_steps["preprocessor"].named_steps["features"]
feat_step.__class__ = EarlySpringFeatures

with open("model.pkl", "wb") as f:
    pickle.dump(final_model, f)

Obviously, randomforests is our winner here, and we will save this as our final model with the best of the hyperparameters we were able to touch on. The test set will be used once and then we wont be able to tune after seeing it.